In [ ]:
import pandas as pd, si_units as si, ternary, matplotlib.pyplot as plt, numpy as np, PLOT_SETTINGS as ps, matplotlib.ticker as ticker, ast, os, feos
from molmass import Formula
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from matplotlib.lines import Line2D
from io import StringIO

In [ ]:
co2_id = feos.Identifier(cas='124-38-9', name='carbon dioxide', iupac_name='carbon dioxide')
carbon_dioxide = feos.PureRecord(identifier=co2_id, molarweight=43.99, m=2.53096, sigma=2.57855, epsilon_k=153.31864)

h2_id = feos.Identifier(cas='1333-74-0', name='hydrogen', iupac_name='molecular hydrogen')
hydrogen = feos.PureRecord(identifier=h2_id, molarweight=2.016, m=1.0, sigma=3.002, epsilon_k=51.343)

n2_id = feos.Identifier(cas='7727-37-9', name='nitrogen', iupac_name='molecular nitrogen')
nitrogen = feos.PureRecord(identifier=n2_id, molarweight=28.006, m=1.23831, sigma=3.30009, epsilon_k=89.41358, nb=2.0)

ar_id = feos.Identifier(cas='7440-37-1', name='argon', iupac_name='argon')
argon = feos.PureRecord(identifier=ar_id, molarweight=39.962, m=1.0, sigma=3.37751, epsilon_k=117.80903)

ch4_id = feos.Identifier(cas='74-82-8', name='methane', iupac_name='methane')
methane = feos.PureRecord(identifier=ch4_id, molarweight=16.031, m=1.0, sigma=3.70051, epsilon_k=150.07147)

o2_id = feos.Identifier(cas='7782-44-7', name='oxygen', iupac_name='molecular oxygen')
oxygen = feos.PureRecord(identifier=o2_id, molarweight=31.99, m=1.14702, sigma=3.17933, epsilon_k=113.62724, nb=2.0)

h2o_id = feos.Identifier(cas='7732-18-5', name='water', iupac_name='oxidane')
water = feos.PureRecord(identifier=h2o_id, molarweight=18.011, m=2.36948, sigma=2.15072, epsilon_k=230.71557, na=1.0, nb=1.0, kappa_ab=0.35319, epsilon_k_ab=2195.10176)

co_id = feos.Identifier(cas='630-08-0', name='carbon monoxide', iupac_name='carbon monoxide')
carbon_monoxide = feos.PureRecord(identifier=co_id, molarweight=27.995, m=1.32286, sigma=3.24532, epsilon_k=91.17087)

no2_id = feos.Identifier(cas='10102-44-0', name='nitrogen dioxide', iupac_name='nitrogen dioxide')
nitrogen_dioxide = feos.PureRecord(identifier=no2_id, molarweight=45.993, m=5.4723, sigma=1.9, epsilon_k=168.28084, nb=2.0)

no_id = feos.Identifier(cas='10102-43-9', name='nitric oxide', iupac_name='nitric oxide')
nitric_oxide = feos.PureRecord(identifier=no_id, molarweight=29.998, m=4.12115, sigma=1.9, epsilon_k=77.07314, nb=1.0)

so2_id = feos.Identifier(cas='7446-09-5', name='sulfur dioxide', iupac_name='sulfur dioxide')
sulfur_dioxide = feos.PureRecord(identifier=so2_id, molarweight=63.962, m=2.69291, sigma=2.73194, epsilon_k=203.03892, mu=1.63, nb=2.0)

h2s_id = feos.Identifier(cas='7783-06-4', name='hydrogen sulfide', iupac_name='sulfane')
hydrogen_sulfide = feos.PureRecord(identifier=h2s_id, molarweight=33.988, m=1.63175, sigma=3.06168, epsilon_k=227.15574, mu=0.97)

c3h8_id = feos.Identifier(cas='74-98-6', name='propane', iupac_name='propane')
propane = feos.PureRecord(identifier=c3h8_id, molarweight=44.063, m=1.98602, sigma=3.6244, epsilon_k=209.08586)

c2h6_id = feos.Identifier(cas='74-84-0', name='ethane', iupac_name='ethane')
ethane = feos.PureRecord(identifier=c2h6_id, molarweight=30.047, m=1.60689, sigma=3.51681, epsilon_k=191.45389)

In [ ]:
def make_parameters(pure_records, kij_pairs=None):
    """
    pure_records : list of feos.PureRecord
    kij_pairs    : list of (record1, record2, k_ij_value), optional
    """
    binary_records = []
    if kij_pairs:
        for rec1, rec2, kij in kij_pairs:
            binary_records.append(feos.BinaryRecord(
                id1=rec1.identifier,
                id2=rec2.identifier,
                k_ij=kij
            ))
    return feos.Parameters.from_records(pure_records, binary_records=binary_records)

def kij_linear(T, k0, k1):
    """Linear k_ij(T) = k0 + k1 * T"""
    return k0 + k1 * T

In [ ]:
T_ref = 300

k1_ar_n2 = -2.0e-5;  k0_ar_n2 = 0.03 - k1_ar_n2 * T_ref  # k0 = 0.036
k1_ar_h2 =  1.5e-5;  k0_ar_h2 = 0.01 - k1_ar_h2 * T_ref  # k0 = 0.0055
k1_n2_h2 = -3.0e-5;  k0_n2_h2 = 0.02 - k1_n2_h2 * T_ref  # k0 = 0.029

kij_ar_n2 = lambda T: k0_ar_n2 + k1_ar_n2 * T
kij_ar_h2 = lambda T: k0_ar_h2 + k1_ar_h2 * T
kij_n2_h2 = lambda T: k0_n2_h2 + k1_n2_h2 * T

In [ ]:
Temp = 300  # K only for KIJ

params_flue = make_parameters(
    [argon, nitrogen, hydrogen],
    kij_pairs=[
        (argon,    nitrogen, kij_ar_n2(Temp)),
        (argon,    hydrogen, kij_ar_h2(Temp)),
        (nitrogen, hydrogen, kij_n2_h2(Temp)),
    ]
)

T               = 100* si.KELVIN
P               = 20e5* si.PASCAL
n_total         = 1.0 * si.MOL
eos             = feos.HelmholtzEnergyFunctional.pcsaft(params_flue)
params_flue

In [ ]:
def save_plot(fig, filename_base, folder="PLOTS"):
    """
    Save a figure as both PNG and PDF using ps.save_figure.
    """
    os.makedirs(folder, exist_ok=True)

    png_path = os.path.join(folder, f"{filename_base}.png")
    pdf_path = os.path.join(folder, f"{filename_base}.pdf")

    ps.save_figure(fig, png_path)
    ps.save_figure(fig, pdf_path)

In [ ]:
H2 = 0.4
n_points = 21

Ar_values = np.linspace(0.0, 1.0 - H2, n_points)

feeds = np.array([
    [Ar, 1.0 - H2 - Ar, H2]
    for Ar in Ar_values
])

print(feeds)


rows = []

for k, z in enumerate(feeds, start=1):
    z = np.asarray(z, dtype=float)
    z = z / z.sum()  # safety: ensure normalized

    feed = z * n_total

    # --- TP flash ---
    try:
        eq = feos.PhaseEquilibrium.tp_flash(eos, T, P, feed)
        liq = eq.liquid
        vap = eq.vapor

        x = np.array(liq.molefracs, dtype=float)
        y = np.array(vap.molefracs, dtype=float)
        
    except Exception as e:
        print(f"TP flash failed for z = {z}: {e}")
        continue

    # --- critical point at this composition ---
    cp = feos.State.critical_point(eos, z*si.MOL, initial_temperature=T)
    critical_temperature = cp.temperature

    # --- planar interface & surface tension ---
    interface = feos.PlanarInterface.from_tanh(
        vle=eq,
        n_grid=500,
        l_grid=100 * si.ANGSTROM,
        critical_temperature=critical_temperature)
    
    try:
        surface_tension         = interface.solve().surface_tension
        interfacial_thickness   = interface.solve().interfacial_thickness
        enrichment              = interface.solve().interfacial_enrichment
        
        gamma_mN_m               = float(surface_tension * 1e3 / si.NEWTON * si.METER)
        interfacial_thickness_nm = float(interfacial_thickness()* 1e9 / si.METER)
        E_1, E_2, E_3            = enrichment()
        
    except Exception as e:
        print(f"Surface tension calculation failed for z = {z}: {e}")
        gamma_mN_m               = np.nan
        interfacial_thickness_nm = np.nan
        E_1, E_2, E_3            = np.nan, np.nan, np.nan

    rows.append({
        "feed_Ar": z[0],
        "feed_N2": z[1],
        "feed_H2": z[2],
        "x_Ar": x[0], "x_N2": x[1], "x_H2": x[2],
        "y_Ar": y[0], "y_N2": y[1], "y_H2": y[2],
        "Tc_K": float(critical_temperature / si.KELVIN),
        "gamma_mN_m": gamma_mN_m,
        "interfacial_thickness_nm": interfacial_thickness_nm,
        "E_Ar": E_1,
        "E_N2": E_2,
        "E_H2": E_3
    })

    print(f"\nCase {k}")
    print("z =", z)
    print("Tc [K] =", float(critical_temperature/si.KELVIN))
    print(f"surface tension [mN/m] = {gamma_mN_m:.6f}")

In [ ]:

x_pts = [tuple(r[c] for c in ["x_Ar","x_N2","x_H2"]) for r in rows]
y_pts = [tuple(r[c] for c in ["y_Ar","y_N2","y_H2"]) for r in rows]

fig, tax = ternary.figure(scale=1.0, permutation='210')
# DEFAULT-ORDER: 012 - BOTTOM, LEFT, RIGHT
fig.set_size_inches(7, 6.5)

tax.boundary(linewidth=2)
tax.gridlines(multiple=0.1, linewidth=0.7)
# tax.convert_coordinates('blr')

tax.left_axis_label(r"Ar $/ \; [\mathrm{mol \; mol^{-1}}]$", offset=0.14, fontsize=20, fontweight="bold")
tax.right_axis_label(r"N$_2$ $/ \; [\mathrm{mol \; mol^{-1}}]$", offset=0.14, fontsize=20, fontweight="bold")
tax.bottom_axis_label(r"H$_2$ $/ \; [\mathrm{mol \; mol^{-1}}]$", offset=0.02, fontsize=20,fontweight="bold")

tax.ticks(axis="lbr",multiple=0.2,linewidth=1,fontsize=11,offset=0.02,tick_formats="%.2f")

tax.scatter(x_pts, marker="o", s=120, facecolors="none", edgecolors="blue", label="Bubble")
tax.scatter(y_pts, marker="o", s=120, facecolors="none", edgecolors="red", label="Dew")
tax.scatter(feeds, marker="o", s=20, facecolors="orange", edgecolors="orange", label="Feed")
# tax.scatter([(0.1, 0.4, 0.50)])

# Optional tie-lines for each case (can clutter if many)
for xp, yp in zip(x_pts, y_pts):
    tax.line(xp, yp, linewidth=0.8, color="k", alpha=0.6, permutation='210')

tax.get_axes().set_axis_off()
tax.clear_matplotlib_ticks()
tax.legend(loc="upper right", frameon=False, fontsize=12)

plt.tight_layout()
plt.show()